# Example Run of LIGNIN-KMC
Written by: Michael Orella <br>
23 January 2019 <br>
Brief examples of how to use the LIGNIN-KMC package within python

In [1]:
%load_ext autoreload
%autoreload 2

Install LIGNIN-KMC

In [2]:
import ligninkmc as kmc
import numpy as np

import scipy.sparse

from ligninkmc.KineticMonteCarlo import doEvent, updateEvents
from ligninkmc.Monomer import Monomer
from ligninkmc.Event import Event

## SG Lignin Example

The first step of this example is for us to define the rates of bond formation. This is done by using a dictionary that maps the bond string ('5o4','bo4','b5','55','bb','ao4',etc...) to a dictionary that maps monomer type (0 - coniferyl, 1 - sinapyl, 2 - caffeoyl) to a dictionary that maps fragment sizes ('monomer','dimer') to the transition state energy in kcal/mol. These values can be tuned and updated to reflect new developments in better understanding of lignin chemistry. Once the values have been input, they are converted from units of kcal/mol to units of joule/molecule. From this point, they are used in the Eyring equation to calculate an equivalent dictionary of dictionaries of dictionaries of rates.

$$ r_i = \dfrac{k_BT}{h}\exp\left(-\dfrac{\Delta G_i}{k_BT}\right) $$

In [3]:
S = 1; G = 0;

In [4]:
kb = 1.38064852e-23 # J / K
h = 6.62607004e-34 # J s
temp = 298.15 #K
kcalToJoule = 4184 / 6.022140857e23

#Energies input in kcal/mol
energies = {'5o4':{(0,0):{('monomer','monomer'):11.2,('monomer','dimer'):14.6,
                          ('dimer','monomer'):14.6,('dimer','dimer'):4.4},
                   (1,0):{('monomer','monomer'):10.9,('monomer','dimer'):14.6,
                          ('dimer','monomer'):14.6,('dimer','dimer'):4.4}},
            '55':{(0,0):{('monomer','monomer'):12.5,('monomer','dimer'):15.6,
                          ('dimer','monomer'):15.6,('dimer','dimer'):3.8}},
            'b5':{(0,0):{('monomer','monomer'):5.5,('monomer','dimer'):5.8,
                          ('dimer','monomer'):5.8,('dimer','dimer'):5.8},
                  (0,1):{('monomer','monomer'):5.5,('monomer','dimer'):5.8,
                          ('dimer','monomer'):5.8,('dimer','dimer'):5.8}},
            'bb':{(0,0):{('monomer','monomer'):5.2,('monomer','dimer'):5.2,('dimer','monomer'):5.2,('dimer','dimer'):5.2},
                  (1,0):{('monomer','monomer'):6.5,('monomer','dimer'):6.5,('dimer','monomer'):6.5,('dimer','dimer'):6.5},
                  (1,1):{('monomer','monomer'):5.2,('monomer','dimer'):5.2,('dimer','monomer'):5.2,('dimer','dimer'):5.2}},
            'bo4':{(0,0):{('monomer','monomer'):6.3,('monomer','dimer'):6.2,
                          ('dimer','monomer'):6.2,('dimer','dimer'):6.2},
                   (1,0):{('monomer','monomer'):9.1,('monomer','dimer'):6.2,
                          ('dimer','monomer'):6.2,('dimer','dimer'):6.2},
                   (0,1):{('monomer','monomer'):8.9,('monomer','dimer'):6.2,
                          ('dimer','monomer'):6.2,('dimer','dimer'):6.2},
                   (1,1):{('monomer','monomer'):9.8,('monomer','dimer'):10.4,
                          ('dimer','monomer'):10.4}},
            'ao4':{(0,0):{('monomer','monomer'):20.7,('monomer','dimer'):20.7,
                          ('dimer','monomer'):20.7,('dimer','dimer'):20.7},
                   (1,0):{('monomer','monomer'):20.7,('monomer','dimer'):20.7,
                          ('dimer','monomer'):20.7,('dimer','dimer'):20.7},
                   (0,1):{('monomer','monomer'):20.7,('monomer','dimer'):20.7,
                          ('dimer','monomer'):20.7,('dimer','dimer'):20.7},
                   (1,1):{('monomer','monomer'):20.7,('monomer','dimer'):20.7,
                          ('dimer','monomer'):20.7,('dimer','dimer'):20.7}},
            'b1':{(0,0):{('monomer','dimer'):9.6,
                          ('dimer','monomer'):9.6,('dimer','dimer'):9.6},
                  (1,0):{('monomer','dimer'):11.7,
                          ('dimer','monomer'):11.7,('dimer','dimer'):11.7},
                  (0,1):{('monomer','dimer'):10.7,
                          ('dimer','monomer'):10.7,('dimer','dimer'):10.7},
                  (1,1):{('monomer','dimer'):11.9,
                          ('dimer','monomer'):11.9,('dimer','dimer'):11.9}},
            'ox':{0:{'monomer':0.9,'dimer':6.3},1:{'monomer':0.6,'dimer':2.2}},
            'q':{0:{'monomer':11.1,'dimer':11.1},1:{'monomer':11.7,'dimer':11.7}}}
energies['bb'][(G,S)] = energies['bb'][(S,G)]

#Correct units of the energies
energiesj = {bond : {monType : {size : energies[bond][monType][size] * kcalToJoule for size in energies[bond][monType]}
                    for monType in energies[bond] } 
            for bond in energies }

#Calculate the rates of reaction
rates = {bond : {monType : { size : kb * temp / h * np.exp ( - energiesj[bond][monType][size] / kb / temp ) 
                            for size in energies[bond][monType]} 
                 for monType in energies[bond] }
         for bond in energies}

With the rates generated, it is now time to initialize the simulation with a set of monomers, and the ratio of S and G units that should be incorporated. 

In [68]:
n = 2

mons = [Monomer(S, i) for i in range(n)]
startEvents = [Event('ox', [i], rates['ox'][S]['monomer']) for i in range(n)]

When the monomers and starting events have been initialized, they are grouped into the "state" and "events" which are necessary to start the simulation. The final pieces of information needed to run the simulation are the maximum number of monomers that should be studied and the final simulation time. In this case, we choose a simulation time of 1 second, and a maximum number of monomers of 10. Once these parameters are set, we can go about running the simulation. To show the state, we will print the adjacency matrix that has been generated, although this is not the typical output examined.

In [69]:
state = { i : {'mon' : mons[i] , 'affected' : {startEvents[i]} } for i in range(n) }
events = { startEvents[i] for i in range(n) }

event_dict = {hash(event): event for event in events}
r_vec = {hash(event): event.rate / n for event in events}
adj = scipy.sparse.dok_matrix((n,n))
t = [0,]
np.random.seed(0)

Check running the Gillespie algorithm by hand to make sure that the proper events are being generated

In [70]:
hashes = list(r_vec.keys())
all_rates = list(r_vec.values())
rtot = np.sum(all_rates)

j = np.random.choice(hashes,p = all_rates/rtot)
event = event_dict[j]

print(event)

dt = ( 1 / rtot ) * np.log ( 1 / np.random.rand(1) )
t.extend(t[-1] + dt)

doEvent(event,state,adj)

Forming ox bond between [1]()



In [71]:
print([str(state[i]["mon"]) for i in state])

['0: sinapyl alcohol is connected to {0} and active at 0', '1: sinapyl alcohol is connected to {1} and active at 4']


In [72]:
updateEvents(state, adj, event, event_dict, r_vec, rates, maxMon=2)

Updating events based on Forming ox bond between [1]()
 with hash value 94072
Before removing this event, the dictionary is {28536: Forming ox bond between [0]()
, 94072: Forming ox bond between [1]()
} with rates {28536: 1128310759786.2012, 94072: 1128310759786.2012}
After removing this event, the dictionary is {28536: Forming ox bond between [0]()
} with rates {28536: 1128310759786.2012}
Possible bonding partners are {'bo4': set(), 'b5': set(), '5o4': set(), '55': set(), 'bb': set(), 'b1': set(), 'ao4': set()}
Modified events = {Forming ox bond between [1]()
}
After removal of any affected events the events are {28536: Forming ox bond between [0]()
}


In [73]:
print(list(event_dict.values()))

[Forming ox bond between [0]()
]


In [74]:
event_to_do = list(event_dict.values())[0]
doEvent(event_to_do, state, adj)

In [75]:
print([str(state[i]["mon"]) for i in state])

['0: sinapyl alcohol is connected to {0} and active at 4', '1: sinapyl alcohol is connected to {1} and active at 4']


In [76]:
updateEvents(state, adj, event_to_do, event_dict, r_vec, rates, maxMon=2)

Updating events based on Forming ox bond between [0]()
 with hash value 28536
Before removing this event, the dictionary is {28536: Forming ox bond between [0]()
} with rates {28536: 1128310759786.2012}
After removing this event, the dictionary is {} with rates {}
Possible bonding partners are {'bo4': {1: sinapyl alcohol 
}, 'b5': {1: sinapyl alcohol 
}, '5o4': {1: sinapyl alcohol 
}, '55': {1: sinapyl alcohol 
}, 'bb': {1: sinapyl alcohol 
}, 'b1': {1: sinapyl alcohol 
}, 'ao4': set()}
Modified events = {Forming ox bond between [0]()
}
After removal of any affected events the events are {}
The first event (this->other) is Forming bo4 bond between [0, 1](4, 8)
 with hash 1522284115000
The first event (other->this) is Forming bo4 bond between [1, 0](8, 4)
 with hash 11417888766004
The second event (this->other) is Forming bo4 bond between [0, 1](8, 4)
 with hash 1522284116020
The second event (other->this) is Forming bo4 bond between [1, 0](4, 8)
 with hash 11417888764984
The first even

In [77]:
print(list(event_dict.values()))

[Forming bo4 bond between [1, 0](4, 8)
, Forming bb bond between [1, 0](8, 8)
, Forming bo4 bond between [1, 0](8, 4)
, Forming bo4 bond between [0, 1](8, 4)
, Forming bo4 bond between [0, 1](4, 8)
, Forming bb bond between [0, 1](8, 8)
]
